# XGBoost - SOJA3 (Com Validacao Cruzada Estratificada)
## Analise comparativa de modelos de Machine Learning - Agro Brasil - TCC

Este notebook implementa modelos XGBoost para prever se o preco de fechamento das acoes SOJA3 ira subir ou nao em diferentes horizontes temporais (3, 7, 15 e 30 dias).

**Dataset utilizado:** Dataset com indicadores de mercado e tecnicos.

**Target:** Classificacao binaria (1 = Alta, 0 = Baixa/Estavel)

**Metodologia:** Train (50%) + Validacao Cruzada Estratificada (30%, 5 folds) + Teste Final (20%)

Sem separacao temporal - Estratificacao mantém proporcoes de classes.

### Importacao das bibliotecas e carregamento do dataset

In [ ]:
import pandas as pd
import numpy as np
from typing import cast
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve,
                             precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
import os

warnings.filterwarnings('ignore')

# Configuracao para salvar graficos
OUTPUT_DIR = './'
os.makedirs(OUTPUT_DIR, exist_ok=True)

COMPANY = 'SOJA3'
DATASET_LABEL = 'com_validacao_cruzada'
DATASET_NAME = 'Indicadores com Validacao Cruzada'
company_lower = COMPANY.lower()

# Carrega dataset CONSOLIDADO
df = pd.read_csv('../../../datasets/datasets_indicadores/classificacao/SOJA3_tratado.csv', 
                  index_col=0, parse_dates=True)

print('='*60)
print(f'DATASET {COMPANY} - XGBOOST ({DATASET_LABEL.upper()})')
print('='*60)
print(f'\nDataset: indicadores_de_mercado_mais_indicadores_agro.csv')
print(f'Shape: {df.shape}')
print('\nColunas disponiveis:')
print(df.columns.tolist())
print('\nPrimeiras 5 linhas:')
print(df.head())
print('\nDistribuicao das classes target:')
for col in ['target_3d', 'target_7d', 'target_15d', 'target_30d']:
    if col in df.columns:
        dist = df[col].value_counts()
        pct = df[col].mean()
        print(f'  {col}: {dist.to_dict()} (Taxa de alta: {pct:.1%})')

print('\nImputacao de ausentes sera feita DENTRO do Pipeline (SimpleImputer), apos o split.')

### Preparacao dos dados para modelagem

In [ ]:
# Definir features: base + indicadores tecnicos + lags
base_features = ['Close', 'Low', 'High', 'Open']
technical_indicators = ['OBV', 'FWMA', 'TEMA', 'HLC3', 'BB_upper', 'BB_middle', 'BB_lower']
lag_features = ['agro_cambio_close_lag_1d', 'agro_cambio_close_lag_3d', 'agro_cambio_close_lag_6d', 'agro_cambio_close_lag_10d',
                'agro_soja_close_lag_1d', 'agro_soja_close_lag_3d', 'agro_soja_close_lag_6d', 'agro_soja_close_lag_10d']
all_features = base_features + technical_indicators + lag_features

X = df[all_features].copy()

# Definir targets (variaveis dependentes) - classificacao binaria
targets = {
    '3d': df['target_3d'],
    '7d': df['target_7d'],
    '15d': df['target_15d'],
    '30d': df['target_30d']
}

print('Variaveis independentes (X):')
print(X.columns.tolist())
print(f'\nShape de X: {X.shape}')
print(f'\nComposicao de features:')
print(f'  - Base features ({len(base_features)}): {base_features}')
print(f'  - Technical indicators ({len(technical_indicators)}): {technical_indicators}')
print(f'  - Lag features ({len(lag_features)}): {lag_features}')
print(f'  - Total: {len(all_features)} features')

print('\nTargets (classificacao binaria):')
for name, target in targets.items():
    print(f'  - {name}: {target.name} (1=Alta, 0=Baixa)')

### Configuracao da Validacao Cruzada Estratificada

In [ ]:
# Definir CV estratificada: 5 folds mantendo proporcao de classes
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Definir espaco de hiperparametros para busca aleatoria
param_dist = {
    'model__n_estimators': [100, 150, 200, 250, 300, 400, 500],
    'model__max_depth': [3, 4, 5, 6, 7],
    'model__learning_rate': [0.01, 0.02, 0.03, 0.05, 0.07, 0.1],
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 2, 3, 5],
    'model__reg_lambda': [0.1, 0.5, 1.0, 2.0, 5.0],
    'model__reg_alpha': [0, 0.1, 0.5, 1.0]
}

print('='*60)
print('CONFIGURACAO DA VALIDACAO CRUZADA')
print('='*60)
print(f'\nEstrategia CV: StratifiedKFold')
print(f'  - N splits (folds): 5')
print(f'  - Shuffle: True')
print(f'  - Random state: 42')

print(f'\nEspaco de hiperparametros (XGB dentro do Pipeline):')
print(f'  - n_estimators: {param_dist["model__n_estimators"]}')
print(f'  - max_depth: {param_dist["model__max_depth"]}')
print(f'  - learning_rate: {param_dist["model__learning_rate"]}')
print(f'  - subsample: {param_dist["model__subsample"]}')
print(f'  - colsample_bytree: {param_dist["model__colsample_bytree"]}')
print(f'  - min_child_weight: {param_dist["model__min_child_weight"]}')
print(f'  - reg_lambda: {param_dist["model__reg_lambda"]}')
print(f'  - reg_alpha: {param_dist["model__reg_alpha"]}')

print(f'\nScoring metric: f1 (foco direto na classe positiva)')

print('\n' + '='*70)
print('DIAGNOSTICO DE DESBALANCEAMENTO POR HORIZONTE')
print('='*70)

for period, y in targets.items():
    y_clean = y.dropna()
    dist = y_clean.value_counts().sort_index()

    n0 = int(dist.get(0, 0))
    n1 = int(dist.get(1, 0))
    total = n0 + n1

    if total == 0:
        print(f'\n{period.upper()}: sem dados')
        continue

    p0 = n0 / total
    p1 = n1 / total

    major = max(n0, n1)
    minor = min(n0, n1)

    if minor == 0:
        imbalance_ratio = np.inf
        ratio_txt = 'infinito (apenas uma classe)'
    else:
        imbalance_ratio = major / minor
        ratio_txt = f'{imbalance_ratio:.2f}:1'

    baseline_acc = major / total

    print(f'\nHorizonte {period.upper()}')
    print(f'  Classe 0 (Baixa/Estavel): {n0} ({p0:.1%})')
    print(f'  Classe 1 (Alta):          {n1} ({p1:.1%})')
    print(f'  Razao majoritaria/minoritaria: {ratio_txt}')
    print(f'  Baseline ingenua (sempre majoritaria): Accuracy = {baseline_acc:.4f}')

    if minor == 0:
        print('  ALERTA: target invalido para classificacao (so uma classe).')
    elif imbalance_ratio >= 3:
        print('  ALERTA: desbalanceamento relevante (>= 3:1).')
    else:
        print('  OK: distribuicao relativamente equilibrada (< 3:1).')

### Treinamento dos Modelos XGBoost com RandomizedSearchCV

In [ ]:
print('TREINAMENTO XGBOOST - SOJA3 (COM VALIDACAO CRUZADA ESTRATIFICADA)')
print('='*60)

models = {}
search_results = {}

for period, y in targets.items():
    print(f'\n\n{"="*60}')
    print(f'PERIODO: {period.upper()}')
    print(f'{"="*60}')
    
    # Remover NaN do target e manter apenas colunas de features definidas
    mask = ~y.isnull()
    X_clean = X.loc[mask].copy()
    y_clean = y.loc[mask].copy()
    
    print(f'\n1. SEPARACAO INICIAL (Train+Val vs Teste Final)')
    print(f'   Dados elegiveis: {X_clean.shape[0]} amostras')
    
    # Step 1: Separar teste final (holdout estratificado)
    X_dev, X_test, y_dev, y_test = train_test_split(
        X_clean, y_clean, 
        test_size=0.2, 
        stratify=y_clean,
        random_state=42, 
        shuffle=True
    )
    
    print(f'   - Desenvolvimento (Train+Val): {X_dev.shape[0]} amostras')
    print(f'   - Teste Final (Holdout): {X_test.shape[0]} amostras')
    print(f'   - Proporcao classe 1 Dev: {y_dev.mean():.1%}')
    print(f'   - Proporcao classe 1 Teste: {y_test.mean():.1%}')
    
    print(f'\n2. BUSCA DE HIPERPARAMETROS (RandomizedSearchCV + StratifiedKFold)')
    
    # Step 2: Pipeline com imputacao para evitar data leakage
    pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
            verbosity=0
        ))
    ])
    
    # Step 3: RandomizedSearchCV com StratifiedKFold
    random_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=50,
        scoring='f1',
        cv=cv_strategy,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    print(f'   - N iteracoes (combinacoes aleatorias): 50')
    print(f'   - CV folds: 5')
    print(f'   - Metrica de selecao: f1 (classe positiva)')
    print(f'   - Jobs paralelos: -1 (todos os cores)')
    print(f'   - Treinando...')
    
    random_search.fit(X_dev, y_dev)
    
    best_pipeline = cast(Pipeline, random_search.best_estimator_)
    best_params = random_search.best_params_
    best_cv_score = random_search.best_score_
    
    print(f'\n   OK - Busca concluida!')
    print(f'   - Melhor F1 (CV): {best_cv_score:.4f}')
    print(f'   - Melhores parametros:')
    for param, value in sorted(best_params.items()):
        print(f'       {param}: {value}')
    
    print(f'\n3. CALIBRACAO DE THRESHOLD (Precision-Recall no conjunto de validacao CV)')
    
    # Out-of-fold probabilities em X_dev para calibrar threshold sem usar o teste final
    oof_pred_proba = cross_val_predict(
        best_pipeline,
        X_dev,
        y_dev,
        cv=cv_strategy,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]
    
    precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_dev, oof_pred_proba)
    f1_vals = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-12)
    
    # precision/recall possuem 1 elemento a mais que thresholds
    if len(pr_thresholds) > 0:
        best_idx = np.argmax(f1_vals[:-1])
        optimal_threshold = float(pr_thresholds[best_idx])
        val_precision = float(precision_vals[best_idx])
        val_recall = float(recall_vals[best_idx])
        val_f1 = float(f1_vals[best_idx])
    else:
        optimal_threshold = 0.5
        val_precision = precision_score(y_dev, (oof_pred_proba >= 0.5).astype(int), zero_division=0)
        val_recall = recall_score(y_dev, (oof_pred_proba >= 0.5).astype(int), zero_division=0)
        val_f1 = f1_score(y_dev, (oof_pred_proba >= 0.5).astype(int), zero_division=0)
    
    print(f'   - Threshold otimo (PR): {optimal_threshold:.4f}')
    print(f'   - Precision validacao (OOF): {val_precision:.4f}')
    print(f'   - Recall validacao (OOF): {val_recall:.4f}')
    print(f'   - F1 validacao (OOF): {val_f1:.4f}')
    
    print(f'\n4. AVALIACAO NO TESTE FINAL (holdout) COM THRESHOLD CALIBRADO')
    
    # Refit final no conjunto de desenvolvimento completo e avaliacao no teste
    best_pipeline.fit(X_dev, y_dev)
    y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= optimal_threshold).astype(int)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    precision_alta = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    recall_alta = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1_alta = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    
    auc_roc = roc_auc_score(y_test, y_pred_proba)
    
    print(f'   Metricas no Teste Final (20%) com threshold calibrado:')
    print(f'   - Threshold aplicado: {optimal_threshold:.4f}')
    print(f'   - Accuracy: {accuracy:.4f}')
    print(f'   - Precision (weighted): {precision:.4f}')
    print(f'   - Recall (weighted): {recall:.4f}')
    print(f'   - F1-Score (weighted): {f1:.4f}')
    print(f'   - AUC-ROC: {auc_roc:.4f}')
    print(f'   ')
    print(f'   Metricas Classe Alta no Teste Final:')
    print(f'   - Precision: {precision_alta:.4f}')
    print(f'   - Recall: {recall_alta:.4f}')
    print(f'   - F1-Score: {f1_alta:.4f}')
    
    models[period] = {
        'model': best_pipeline,
        'best_params': best_params,
        'best_cv_score': best_cv_score,
        'optimal_threshold': optimal_threshold,
        'val_precision_oof': val_precision,
        'val_recall_oof': val_recall,
        'val_f1_oof': val_f1,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_alta': precision_alta,
        'recall_alta': recall_alta,
        'f1_alta': f1_alta,
        'auc_roc': auc_roc,
        'X_dev': X_dev,
        'X_test': X_test,
        'y_dev': y_dev,
        'y_test': y_test,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'oof_pred_proba': oof_pred_proba,
        'search_object': random_search
    }
    
    search_results[period] = {
        'best_cv_score': best_cv_score,
        'best_params': best_params,
        'optimal_threshold': optimal_threshold,
        'oof_f1_at_optimal_threshold': val_f1,
        'test_accuracy': accuracy,
        'test_f1': f1,
        'test_f1_alta': f1_alta
    }

print(f'\n\n{"="*60}')
print('OK - 4 modelos treinados com validacao cruzada e threshold calibrado!')
print('='*60)

### Tabela Resumo das Metricas (Teste Final)

In [ ]:
# Criar DataFrame com resumo das metricas
metrics_summary = pd.DataFrame({
    'Horizonte': ['3 dias', '7 dias', '15 dias', '30 dias'],
    'CV F1': [models['3d']['best_cv_score'], models['7d']['best_cv_score'], 
              models['15d']['best_cv_score'], models['30d']['best_cv_score']],
    'Threshold Otimo (PR)': [models['3d']['optimal_threshold'], models['7d']['optimal_threshold'],
                             models['15d']['optimal_threshold'], models['30d']['optimal_threshold']],
    'OOF F1 no Threshold': [models['3d']['val_f1_oof'], models['7d']['val_f1_oof'],
                            models['15d']['val_f1_oof'], models['30d']['val_f1_oof']],
    'Test Accuracy': [models['3d']['accuracy'], models['7d']['accuracy'], 
                      models['15d']['accuracy'], models['30d']['accuracy']],
    'Test Precision': [models['3d']['precision'], models['7d']['precision'], 
                       models['15d']['precision'], models['30d']['precision']],
    'Test Recall': [models['3d']['recall'], models['7d']['recall'], 
                    models['15d']['recall'], models['30d']['recall']],
    'Test F1-Score': [models['3d']['f1'], models['7d']['f1'], 
                      models['15d']['f1'], models['30d']['f1']],
    'Test F1 Classe Alta': [models['3d']['f1_alta'], models['7d']['f1_alta'],
                            models['15d']['f1_alta'], models['30d']['f1_alta']],
    'Test AUC-ROC': [models['3d']['auc_roc'], models['7d']['auc_roc'], 
                     models['15d']['auc_roc'], models['30d']['auc_roc']]
})

print('\nRESUMO DAS METRICAS - SOJA3 (COM VALIDACAO CRUZADA)')
print('='*120)
print(metrics_summary.to_string(index=False))
print('\nLegenda:')
print('  - CV F1: F1 medio da validacao cruzada (5 folds) no RandomizedSearchCV')
print('  - Threshold Otimo (PR): limiar calibrado via curva Precision-Recall em previsoes OOF do Dev')
print('  - OOF F1 no Threshold: desempenho de validacao com o limiar calibrado')
print('  - Test *: metricas no teste final (20% holdout) aplicando threshold calibrado')

# Salvar metricas em CSV
metrics_summary.to_csv(f'{OUTPUT_DIR}metricas_{company_lower}_{DATASET_LABEL}.csv', index=False)
print(f'\nOK - Metricas salvas em metricas_{company_lower}_{DATASET_LABEL}.csv')

### Melhores Parametros Encontrados

In [ ]:
print('MELHORES PARAMETROS ENCONTRADOS POR HORIZONTE')
print('='*80)

for period in ['3d', '7d', '15d', '30d']:
    print(f'\n{period.upper()}:')
    print(f'  CV F1-Score: {models[period]["best_cv_score"]:.4f}')
    print(f'  Parametros:')
    for param, value in sorted(models[period]['best_params'].items()):
        print(f'    - {param}: {value}')

print('\n' + '='*80)

---
# Analises Graficas

### Matrizes de Confusao (Teste Final)

In [ ]:
# Matrizes de Confusao
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (period, model_data) in enumerate(models.items()):
    y_test = model_data['y_test']
    y_pred = model_data['y_pred']

    cm = confusion_matrix(y_test, y_pred)

    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Baixa/Estavel', 'Alta'],
                yticklabels=['Baixa/Estavel', 'Alta'],
                cbar_kws={'label': 'Amostras'})

    ax.set_title(f'Matriz de Confusao {period.upper()} - {COMPANY}\nAccuracy (Teste): {model_data["accuracy"]:.4f} | CV F1: {model_data["best_cv_score"]:.4f}', fontweight='bold')
    ax.set_xlabel('Predicao')
    ax.set_ylabel('Valor Real')

    # Adicionar porcentagens
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            pct = (cm[i, j] / total) * 100
            ax.text(j + 0.5, i + 0.7, f'({pct:.1f}%)', ha='center', va='center', fontsize=9, color='red')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}matriz_confusao_{company_lower}_{DATASET_LABEL}.png', dpi=300, bbox_inches='tight')
plt.show()
print('OK - Matrizes de confusao salvas')

### Curvas ROC (Teste Final)

In [ ]:
# Curvas ROC
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (period, model_data) in enumerate(models.items()):
    y_test = model_data['y_test']
    y_pred_proba = model_data['y_pred_proba']
    auc = model_data['auc_roc']

    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

    ax = axes[idx]
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc:.4f})')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Linha Base')
    ax.fill_between(fpr, tpr, alpha=0.2)

    ax.set_title(f'Curva ROC {period.upper()} - {COMPANY} (Teste Final)', fontweight='bold')
    ax.set_xlabel('Taxa de Falsos Positivos (FPR)')
    ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}curva_roc_{company_lower}_{DATASET_LABEL}.png', dpi=300, bbox_inches='tight')
plt.show()
print('OK - Curvas ROC salvas')

### Comparacao: CV Score vs Test Accuracy

In [ ]:
# Comparacao CV vs Teste
periods = ['3d', '7d', '15d', '30d']

cv_scores = [models[p]['best_cv_score'] for p in periods]
test_acc = [models[p]['accuracy'] for p in periods]
test_f1 = [models[p]['f1'] for p in periods]
test_auc = [models[p]['auc_roc'] for p in periods]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV vs Teste
x = np.arange(len(periods))
width = 0.35

bars1 = axes[0].bar(x - width/2, cv_scores, width, label='CV F1-Score', color='#3498db')
bars2 = axes[0].bar(x + width/2, test_acc, width, label='Test Accuracy', color='#e74c3c')

axes[0].set_xlabel('Horizonte Temporal')
axes[0].set_ylabel('Score')
axes[0].set_title(f'Estabilidade: CV F1-Score vs Test Accuracy', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(periods)
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3, axis='y')

# Metricas Teste
axes[1].plot(periods, test_acc, marker='o', linewidth=2, markersize=10, label='Accuracy', color='#2ecc71')
axes[1].plot(periods, test_f1, marker='s', linewidth=2, markersize=10, label='F1-Score', color='#e74c3c')
axes[1].plot(periods, test_auc, marker='^', linewidth=2, markersize=10, label='AUC-ROC', color='#f39c12')
axes[1].set_title(f'Metricas no Teste Final (20%)', fontweight='bold')
axes[1].set_xlabel('Horizonte')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}comparacao_cv_vs_teste_{company_lower}_{DATASET_LABEL}.png', dpi=300, bbox_inches='tight')
plt.show()
print('OK - Grafico de comparacao CV vs Teste salvo')

### Metricas da Classe Alta (Teste Final)

In [ ]:
# Metricas especificas da classe Alta
precision_alta_vals = [models[p]['precision_alta'] for p in periods]
recall_alta_vals = [models[p]['recall_alta'] for p in periods]
f1_alta_vals = [models[p]['f1_alta'] for p in periods]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(periods))
width = 0.25

bars1 = ax.bar(x - width, precision_alta_vals, width, label='Precision (Alta)', color='#3498db')
bars2 = ax.bar(x, recall_alta_vals, width, label='Recall (Alta)', color='#9b59b6')
bars3 = ax.bar(x + width, f1_alta_vals, width, label='F1-Score (Alta)', color='#e74c3c')

ax.set_xlabel('Horizonte Temporal')
ax.set_ylabel('Score')
ax.set_title(f'Metricas da Classe Alta (Positiva) - Teste Final (20%)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(periods)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Adicionar valores nas barras
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}metricas_classe_alta_{company_lower}_{DATASET_LABEL}.png', dpi=300, bbox_inches='tight')
plt.show()
print('OK - Metricas da classe Alta salvas')

### Relatorio de Classificacao Detalhado (Teste Final)

In [ ]:
# Relatorio de classificacao detalhado
print('\n' + '='*60)
print(f'RELATORIO DE CLASSIFICACAO DETALHADO - TESTE FINAL')
print('='*60)

for period, model_data in models.items():
    print('\n' + '-' * 80)
    print(f'HORIZONTE: {period.upper()}')
    print('-' * 80)
    print(classification_report(model_data['y_test'], model_data['y_pred'],
                                target_names=['Baixa/Estavel', 'Alta']))

### Interpretabilidade com SHAP (Magnitude e Direcao do Impacto)

In [ ]:
periods = ['3d', '7d', '15d', '30d']
shap_rankings = {}

for period in periods:
    print(f'\nGerando SHAP summary plot para {period.upper()}...')
    
    pipeline_model = models[period]['model']
    imputer = pipeline_model.named_steps['imputer']
    xgb_model = pipeline_model.named_steps['model']
    X_shap_raw = models[period]['X_test'].copy()
    
    # Aplicar a mesma imputacao aprendida no treino antes de calcular SHAP
    X_shap_imputed = imputer.transform(X_shap_raw)
    X_shap_df = pd.DataFrame(X_shap_imputed, columns=all_features, index=X_shap_raw.index)
    
    # Prepara dados de background do conjunto de desenvolvimento para referencia do SHAP
    X_dev_imputed = imputer.transform(models[period]['X_dev'])
    X_dev_df = pd.DataFrame(X_dev_imputed, columns=all_features, index=models[period]['X_dev'].index)
    # Amostra aleatória de dados de treino como background (até 100 amostras)
    background_sample_size = min(100, max(50, X_dev_df.shape[0] // 10))
    background_indices = np.random.choice(X_dev_df.shape[0], size=background_sample_size, replace=False)
    X_background = X_dev_df.iloc[background_indices]
    
    try:
        # Tentar com TreeExplainer primeiro (mais eficiente)
        print(f'   - Tentando TreeExplainer...')
        explainer = shap.TreeExplainer(xgb_model, data=X_background)
        shap_values = explainer.shap_values(X_shap_df)
        print(f'   - OK: TreeExplainer funcionou')
    except (ValueError, Exception) as e1:
        try:
            # Fallback 1: KernelExplainer com função predict_proba customizada
            print(f'   - TreeExplainer falhou ({type(e1).__name__}), tentando KernelExplainer...')
            
            # Função wrapper que aplica imputation + prediction
            def predict_fn(X):
                X_imp = imputer.transform(X)
                return xgb_model.predict_proba(X_imp)[:, 1]
            
            explainer = shap.KernelExplainer(predict_fn, X_background)
            shap_values = explainer.shap_values(X_shap_df)
            print(f'   - OK: KernelExplainer funcionou')
        except Exception as e2:
            try:
                # Fallback 2: Permutation explainer (mais lento mas muito robusto)
                print(f'   - KernelExplainer falhou ({type(e2).__name__}), tentando PermutationExplainer...')
                
                # Função wrapper que aplica imputation + prediction
                def predict_fn_perm(X):
                    if isinstance(X, pd.DataFrame):
                        X_imp = imputer.transform(X)
                    else:
                        X_imp = imputer.transform(pd.DataFrame(X, columns=all_features))
                    proba = xgb_model.predict_proba(X_imp)
                    return proba[:, 1]
                
                explainer = shap.PermutationExplainer(predict_fn_perm, X_background)
                shap_values = explainer.shap_values(X_shap_df)
                print(f'   - OK: PermutationExplainer funcionou')
            except Exception as e3:
                print(f'   - ERRO: Todos os explainers falharam. Usando feature importance nativa.')
                print(f'      TreeExplainer: {type(e1).__name__}')
                print(f'      KernelExplainer: {type(e2).__name__}')
                print(f'      PermutationExplainer: {type(e3).__name__}')
                
                # Fallback final: usar feature_importances_ do modelo
                mean_abs_shap = xgb_model.feature_importances_
                ranking_df = pd.DataFrame({
                    'Variavel': all_features,
                    'mean(|SHAP|)': mean_abs_shap
                }).sort_values(by='mean(|SHAP|)', ascending=False)
                shap_rankings[period] = ranking_df
                
                print(f'Top 10 variaveis por Feature Importance - {period.upper()}:')
                print(ranking_df.head(10).to_string(index=False))
                continue
    
    # Se shap_values for lista (multi-class), pegar classe positiva (índice 1)
    if isinstance(shap_values, list):
        shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
        print(f'   - Usando SHAP values da classe positiva (índice 1)')
    
    # Converter para numpy array se necessário
    if not isinstance(shap_values, np.ndarray):
        shap_values = np.array(shap_values)
    
    # Summary plot (beeswarm): mostra magnitude e direcao do impacto por variavel
    plt.figure(figsize=(12, 7))
    shap.summary_plot(shap_values, X_shap_df, feature_names=all_features, max_display=10, show=False)
    plt.title(f'SHAP Summary Plot - {COMPANY} {period.upper()} (Teste Final)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}shap_summary_{company_lower}_{period}_{DATASET_LABEL}.png', dpi=300, bbox_inches='tight')
    plt.close()
    plt.show()
    
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    ranking_df = pd.DataFrame({
        'Variavel': all_features,
        'mean(|SHAP|)': mean_abs_shap
    }).sort_values(by='mean(|SHAP|)', ascending=False)
    shap_rankings[period] = ranking_df
    
    print(f'Top 10 variaveis por mean(|SHAP|) - {period.upper()}:')
    print(ranking_df.head(10).to_string(index=False))

print('\nOK - SHAP summary plots gerados para todos os horizontes.')

### Resumo Final

In [ ]:
print('\n' + '='*80)
print(f'RESUMO FINAL - XGBOOST {COMPANY} (COM VALIDACAO CRUZADA ESTRATIFICADA)')
print('='*80)

print('\nMETODOLOGIA:')
print('  1. Train + Validacao (Dev): 80% dos dados')
print('  2. Teste Final (Holdout): 20% dos dados')
print('  3. Dentro do Dev: StratifiedKFold (5 folds) + RandomizedSearchCV (50 iteracoes)')
print('  4. Pipeline anti-leakage: SimpleImputer (mediana) + XGBoost')
print('  5. Metrica de selecao de hiperparametros: F1 (classe positiva)')
print('  6. Calibracao de threshold por Precision-Recall usando previsoes OOF no Dev')

print('\n' + '-'*80)
print('RESULTADOS POR HORIZONTE:')
print('-'*80)
periods = ['3d', '7d', '15d', '30d']
for period in periods:
    m = models[period]
    print(f"\n{period.upper()}:")
    print(f"  CV F1-Score (5 folds):          {m['best_cv_score']:.4f}")
    print(f"  Threshold otimo (PR):           {m['optimal_threshold']:.4f}")
    print(f"  OOF F1 com threshold:           {m['val_f1_oof']:.4f}")
    print(f"  Test Accuracy:                  {m['accuracy']:.4f}")
    print(f"  Test Precision (weighted):      {m['precision']:.4f}")
    print(f"  Test Recall (weighted):         {m['recall']:.4f}")
    print(f"  Test F1-Score (weighted):       {m['f1']:.4f}")
    print(f"  Test AUC-ROC:                   {m['auc_roc']:.4f}")
    print(f"  Classe Alta F1:                 {m['f1_alta']:.4f}")

print('\n' + '-'*80)
print('ARQUIVOS GERADOS:')
print('-'*80)
print(f'  - metricas_{company_lower}_{DATASET_LABEL}.csv')
print(f'  - matriz_confusao_{company_lower}_{DATASET_LABEL}.png')
print(f'  - curva_roc_{company_lower}_{DATASET_LABEL}.png')
print(f'  - comparacao_cv_vs_teste_{company_lower}_{DATASET_LABEL}.png')
print(f'  - metricas_classe_alta_{company_lower}_{DATASET_LABEL}.png')
for period in periods:
    print(f'  - shap_summary_{company_lower}_{period}_{DATASET_LABEL}.png')
print('\n' + '='*80)